In [1]:
# %% [markdown]
# Import necessary libraries
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Subset
import torch.optim as optim
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error
import json
from tqdm import tqdm
import torch.nn.functional as F
import joblib
import os
os.environ['JAX_PLATFORMS'] = 'cpu'
import sys
from torch.utils.data import DataLoader, random_split
import time  # For timing

# Set matplotlib to use a non-interactive backend
plt.switch_backend('Agg')

# Get the notebook's directory
notebook_dir = os.getcwd()
# Add parent directory to path
project_root = os.path.dirname(notebook_dir)
sys.path.append(project_root)

# %% [markdown]
# Checking if the MPS (Metal Performance Shaders) backend is available

# %%
# Check if MPS is available
if not torch.backends.mps.is_available():
    raise RuntimeError("MPS device not available. Check if PyTorch and macOS set up correctly.")

# Set the device to MPS
device = torch.device("mps")  # Use GPU on M2 Pro

In [2]:
# %%
# Import the objective function
# Import the physics engine from the src folder
from src.qkd.physics import (
    calculate_eta_ch, calculate_eta_sys, calculate_D_mu_k, 
    calculate_n_X_total, calculate_N, calculate_n_Z_total,
    calculate_e_mu_k, calculate_e_obs, calculate_h, calculate_lambda_EC,
    calculate_sqrt_term, calculate_n_pm, calculate_S_0, calculate_S_1,
    calculate_m_mu_k, calculate_m_pm, calculate_v_1, calculate_gamma,
    calculate_Phi, calculate_l, calculate_R
)
from src.qkd.model import calculate_key_rates_and_metrics, objective, penalty


In [ ]:
# Define a safe wrapper for the objective function
# def safe_objective(params, L, nx, **kwargs):
#     try:
#         key_rate = objective(params, L, nx, **kwargs)[0]
#         if np.isnan(key_rate) or np.isinf(key_rate) or key_rate <= 0:
#             print(f"Invalid key rate (NaN, Inf, or <= 0) for params {params}, L={L}, nx={nx}: {key_rate}")
#             return None  # Return None to indicate invalid result
#         return key_rate
#     except Exception as e:
#         print(f"Error in objective for params {params}, L={L}, nx={nx}: {e}")
#         return None  # Return None on error

def safe_objective(params, L, nx, **kwargs):
    try:
        # Ensure params are in a numpy array for checking
        params_np = np.array(params)
        # Check for NaN/Inf in input params themselves (can happen from NN)
        if np.any(np.isnan(params_np)) or np.any(np.isinf(params_np)):
             print(f"Invalid input params (NaN/Inf): {params_np}, L={L}, nx={nx}")
             return None

        key_rate = objective(params, L, nx, **kwargs)[0]

        if np.isnan(key_rate):
            print(f"Objective returned NaN for params {params}, L={L}, nx={nx}")
            return None
        if np.isinf(key_rate):
            print(f"Objective returned Inf for params {params}, L={L}, nx={nx}")
            return None
        if key_rate <= 0:
            # Filter non-positive key rates (expected behavior near physical limits)
            print(f"Objective returned non-positive key rate ({key_rate}) for params {params}, L={L}, nx={nx}")
            # Decide if you want to return None or the non-positive value
            return None # Invalid: key rates must be positive
        return key_rate
    except Exception as e:
        print(f"Exception in objective for params {params}, L={L}, nx={nx}: {e}")
        # Add traceback for more detail if needed:
        # import traceback
        # print(traceback.format_exc())
        return None
    
# # %%
# # Load dataset
# with open('../Training_Data/n_X/good/cleaned_combined_datasets.json', 'r') as f:
#     data_by_nx = json.load(f)

# print(f"The overall dataset contains {len(data_by_nx)} entries (number of unique n_X values).")

# # Print the number of entries for each n_X before filtering
# for n_x in data_by_nx.keys():
#     print(f"Number of entries for n_X = {n_x}: {len(data_by_nx[n_x])}")

# Load dataset
import json
with open('Training_Data/n_X/good/cleaned_combined_datasets.json', 'r') as f:
    data_by_nx = json.load(f)

print(f"The overall dataset contains {len(data_by_nx)} entries (number of unique n_X values).")

# Print the number of entries for each n_X before filtering
for n_x in data_by_nx.keys():
    print(f"Number of entries for n_X = {n_x}: {len(data_by_nx[n_x])}")
    

The overall dataset contains 6 entries (number of unique n_X values).
Number of entries for n_X = 10000.0: 736
Number of entries for n_X = 100000.0: 855
Number of entries for n_X = 1000000.0: 902
Number of entries for n_X = 10000000.0: 927
Number of entries for n_X = 100000000.0: 942
Number of entries for n_X = 1000000000.0: 948


In [4]:
# Flatten the data structure and filter
cleaned_data = []
for n_x, entries in data_by_nx.items():
    filtered_entries = [item for item in entries if item["key_rate"] > 0 and item["e_1"] * 100 <= 200]
    print(f"After filtering, n_X = {n_x} has {len(filtered_entries)} entries.")
    cleaned_data.extend(filtered_entries)

# Verify the cleaned dataset
if not cleaned_data:
    print("No valid data after filtering.")
else:
    print(f"Filtered dataset contains {len(cleaned_data)} entries.")
    print("\nSample entry from the cleaned dataset:")
    print(json.dumps(cleaned_data[0], indent=2))
    print("\nNumber of unique n_X values:", len(data_by_nx))

# %%
X = np.array([[item['e_1'], item['e_2'], item['e_3'], item['e_4']] for item in cleaned_data])
Y = np.array([[item['optimized_params']['mu_1'], item['optimized_params']['mu_2'], item['optimized_params']['P_mu_1'], item['optimized_params']['P_mu_2'], item['optimized_params']['P_X_value']] for item in cleaned_data])

After filtering, n_X = 10000.0 has 736 entries.
After filtering, n_X = 100000.0 has 855 entries.
After filtering, n_X = 1000000.0 has 902 entries.
After filtering, n_X = 10000000.0 has 927 entries.
After filtering, n_X = 100000000.0 has 942 entries.
After filtering, n_X = 1000000000.0 has 948 entries.
Filtered dataset contains 5310 entries.

Sample entry from the cleaned dataset:
{
  "fiber_length": 0.0,
  "e_1": 0.0,
  "e_2": 6.221848749616356,
  "e_3": 0.5,
  "e_4": 4.0,
  "key_rate": 0.00037995895580899045,
  "optimized_params": {
    "mu_1": 0.6126301275287198,
    "mu_2": 0.13818683663411718,
    "P_mu_1": 0.04604299826386499,
    "P_mu_2": 0.6110117432731096,
    "P_X_value": 0.414731486913827
  }
}

Number of unique n_X values: 6


In [5]:
from sklearn.utils import shuffle

X, Y = shuffle(X, Y, random_state=42)

scaler = StandardScaler()
X = scaler.fit_transform(X)  # Fit and transform on training data

y_scaler = MinMaxScaler()  # Scale targets to [0, 1]
Y = y_scaler.fit_transform(Y)

# Save the scalers
joblib.dump(scaler, 'models/scaler.pkl')  # Save StandardScaler
joblib.dump(y_scaler, 'models/y_scaler.pkl')  # Save MinMaxScaler

print(f"X shape: {X.shape}, Y shape: {Y.shape}")
dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32))

# %%
# Split dataset into train and validation sets
train_size = int(0.8 * len(dataset))  # 80% for training
val_size = len(dataset) - train_size  # 20% for validation

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Data loaders with increased batch size
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

X shape: (5310, 4), Y shape: (5310, 5)


In [6]:
# %% [markdown]
# ### Prepare Evaluation Data for All n_X Values (10^4 to 10^9) Before Training

# Load the combined dataset
# combined_file_path = '../Training_Data/n_X/good/cleaned_combined_datasets.json'
# try:
#     with open(combined_file_path, 'r') as f:
#         combined_data = json.load(f)
#     print(f"Loaded combined dataset with {len(combined_data)} n_X entries.")
# except FileNotFoundError:
#     raise FileNotFoundError(f"Combined dataset not found at {combined_file_path}.")

# %% [markdown]
# ### Prepare Evaluation Data for All n_X Values (10^4 to 10^9) Before Training

# Load the combined dataset
# Note: Path corrected to be relative to the current notebook location
combined_file_path = 'Training_Data/n_X/good/cleaned_combined_datasets.json'
try:
    with open(combined_file_path, 'r') as f:
        combined_data = json.load(f)
    print(f"Loaded combined dataset with {len(combined_data)} n_X entries.")
except FileNotFoundError:
    raise FileNotFoundError(f"Combined dataset not found at {combined_file_path}.")

Loaded combined dataset with 6 n_X entries.


In [7]:
# Define the range of n_X values to evaluate
nx_values = [10**s for s in range(4, 10)]  # 10^4 to 10^9

# Dictionary to store evaluation data for each n_X
all_evaluation_data = {}

In [8]:
# Extract and prepare evaluation data for each n_X
for nx in nx_values:
    nx_key = str(float(nx))
    if nx_key not in combined_data:
        print(f"No data found for n_X = {nx} in the combined dataset. Skipping...")
        continue
    evaluation_data = combined_data[nx_key]
    print(f"Extracted {len(evaluation_data)} entries for n_X = {nx}.")

    # Extract fiber lengths and optimized parameters
    fiber_lengths = np.array([entry["fiber_length"] for entry in evaluation_data])
    optimized_params_array = np.array([list(entry["optimized_params"].values()) for entry in evaluation_data])

    # Compute optimized key rates
    optimized_key_rates = []
    valid_indices = []
    for idx, (params, L) in enumerate(zip(optimized_params_array, fiber_lengths)):
        key_rate = safe_objective(params, L, nx, alpha=0.2, eta_Bob=0.1, P_dc_value=6e-7, 
                                  epsilon_sec=1e-10, epsilon_cor=1e-15, f_EC=1.16, 
                                  e_mis=5e-3, P_ap=0, n_event=1)
        if key_rate is None:
            print(f"Skipping invalid optimized key rate at index {idx} for n_X = {nx}")
            continue
        optimized_key_rates.append(key_rate)
        valid_indices.append(idx)
    optimized_key_rates = np.array(optimized_key_rates)

     #Filter the evaluation data to exclude invalid key rates
    if len(valid_indices) == 0:
        print(f"Warning: No valid key rates computed for n_X = {nx}. Skipping.")
        continue

    # Filter the evaluation data to exclude invalid key rates
    fiber_lengths = fiber_lengths[valid_indices]
    optimized_params_array = optimized_params_array[valid_indices]

    # Prepare test inputs for evaluation
    X_test = []
    for L in fiber_lengths:
        e_1 = L / 100
        e_2 = -np.log10(6e-7)
        e_3 = 5e-3 * 100
        e_4 = np.log10(nx)
        X_test.append([e_1, e_2, e_3, e_4])
    X_test = np.array(X_test)

    if X_test.size == 0:
         print(f"Warning: X_test is empty for n_X = {nx}. Skipping.")
         continue
        
    X_test_scaled = scaler.transform(X_test)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)

    # Store all evaluation data in a dictionary
    all_evaluation_data[nx] = {
        'fiber_lengths': fiber_lengths,
        'optimized_params_array': optimized_params_array,
        'optimized_key_rates': optimized_key_rates,
        'X_test_tensor': X_test_tensor
    }
    print(f"After filtering invalid key rates, {len(optimized_key_rates)} data points remain for n_X = {nx}.")

Extracted 736 entries for n_X = 10000.
After filtering invalid key rates, 736 data points remain for n_X = 10000.
Extracted 855 entries for n_X = 100000.
After filtering invalid key rates, 855 data points remain for n_X = 100000.
Extracted 902 entries for n_X = 1000000.
After filtering invalid key rates, 902 data points remain for n_X = 1000000.
Extracted 927 entries for n_X = 10000000.
After filtering invalid key rates, 927 data points remain for n_X = 10000000.
Extracted 942 entries for n_X = 100000000.
After filtering invalid key rates, 942 data points remain for n_X = 100000000.
Extracted 948 entries for n_X = 1000000000.
After filtering invalid key rates, 948 data points remain for n_X = 1000000000.


In [9]:
# %%
# Define the neural network model
class BB84NN(nn.Module):
    def __init__(self):
        super(BB84NN, self).__init__()
        self.fc1 = nn.Linear(4, 16)
        self.fc2 = nn.Linear(16, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, 5)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [10]:
# Initialize model, loss, optimizer, and scheduler
model = BB84NN().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, min_lr=1e-6)

# Lists to store metrics
train_losses = []
val_losses = []
learning_rates = []

In [11]:
def plot_keyrate_subplots(all_data, epoch, filename):
    """
    Plots key rates vs fiber length for multiple n_X values.
    Handles cases where predicted data may be sparse or empty (e.g., at early epochs).
    """
    nx_values = sorted(all_data.keys(), key=lambda x: float(x))
    num_plots = len(nx_values)
    cols = 3
    rows = (num_plots + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows), sharex=True, sharey=True)
    fig.suptitle(f"Key Rates vs Fiber Length at Epoch {epoch} ($n_X = 10^s$)", fontsize=16)
    
    # Handle single vs multiple plots
    if num_plots > 1:
        axes = axes.flatten()
    else:
        axes = np.array([axes])

    for i, nx in enumerate(nx_values):
        ax = axes[i]
        
        # Get data (use consistent naming)
        fiber_lengths_opt = all_data[nx]['fiber_lengths']
        optimized_key_rates = all_data[nx]['optimized_key_rates']
        predicted_key_rates = all_data[nx].get('predicted_key_rates', np.array([]))
        fiber_lengths_pred = all_data[nx].get('fiber_lengths_pred', np.array([]))
        
        # ALWAYS plot optimized (ground truth)
        if len(optimized_key_rates) > 0:
            mask_opt = ~np.isnan(optimized_key_rates) & (optimized_key_rates > 0)
            if np.any(mask_opt):
                ax.plot(fiber_lengths_opt[mask_opt], 
                       np.log10(optimized_key_rates[mask_opt]), 
                       'b-', label='Optimized', linewidth=2, alpha=0.9)
    # for i, nx in enumerate(nx_values):
    #     ax = axes[i]
    #     data = all_data[nx]
        
    #     # # Optimized data (always available)
    #     fiber_lengths_opt = data['fiber_lengths']
    #     optimized_key_rates = data['optimized_key_rates']
        
    #     #Predicted data (may be sparse or empty, especially at epoch 0)
    #     predicted_key_rates = data.get('predicted_key_rates', np.array([]))
    #     fiber_lengths_pred = data.get('fiber_lengths_pred', fiber_lengths_opt)
        
    #     # Get data
    #     fiber_lengths_opt = all_data[nx]['fiber_lengths']  # ← For optimized
    #     optimized_key_rates = all_data[nx]['optimized_key_rates']
        
    #     predicted_key_rates = all_data[nx].get('predicted_key_rates', np.array([]))
    #     fiber_lengths_pred = all_data[nx].get('fiber_lengths_pred', np.array([]))  # ← NEW!

        # # Plot optimized (ALWAYS)
        # ax.plot(fiber_lengths_opt, np.log10(optimized_key_rates), 'b-', label='Optimized', linewidth=2.0)
        
        # ALWAYS plot optimized data (the ground truth)
        if len(predicted_key_rates) > 0 and len(fiber_lengths_pred) > 0:
            mask_pred = ~np.isnan(predicted_key_rates) & (predicted_key_rates > 0)
            if np.any(mask_pred):
                ax.plot(fiber_lengths_pred[mask_pred], np.log10(predicted_key_rates[mask_pred]), 'r--', label='Predicted (NN)', linewidth=2.5, alpha=0.7)

        # if len(optimized_key_rates) > 0:
        #     mask_opt = ~np.isnan(optimized_key_rates) & (optimized_key_rates > 0)
        #     if np.any(mask_opt):
        #         ax.plot(fiber_lengths_opt[mask_opt], 
        #                np.log10(optimized_key_rates[mask_opt]), 
        #                'b-', label='Optimized', linewidth=2, alpha=0.9)
        
        # Plot predicted data ONLY if available
        if len(predicted_key_rates) > 0:
            mask_pred = ~np.isnan(predicted_key_rates) & (predicted_key_rates > 0)
            if np.any(mask_pred):
                ax.plot(fiber_lengths_pred[mask_pred], np.log10(predicted_key_rates[mask_pred]), 'r--', label='Predicted (NN)', linewidth=2.5, alpha=0.7)
        
        # Formatting
        exponent = int(np.log10(float(nx)))
        ax.set_title(f"$n_X = 10^{{{exponent}}}$", fontsize=12)
        ax.set_xlabel("Fiber Length (km)")
        ax.set_ylabel("log₁₀(Key Rate)")
        ax.grid(True, which='both', linestyle=':', alpha=0.5)
        
        if i == 0:
            ax.legend(loc='lower left', fontsize=10)

    # Remove empty subplots
    for i in range(num_plots, len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Key rate subplots saved to {filename}")


In [12]:
def plot_parameters_subplots(all_data, epoch, filename):
    """
    Plots parameters vs fiber length for multiple n_X values.
    Handles cases where predicted data may be sparse or empty.
    """
    nx_values = sorted(all_data.keys(), key=lambda x: float(x))
    num_plots = len(nx_values)
    cols = 3
    rows = (num_plots + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows), sharex=True, sharey=False)
    fig.suptitle(f"Optimized Parameters vs Fiber Length at Epoch {epoch} ($n_X = 10^s$)", fontsize=16)
    
    if num_plots > 1:
        axes = axes.flatten()
    else:
        axes = np.array([axes])
    
    param_labels = ['$\\mu_1$', '$\\mu_2$', '$P_{\\mu_1}$', '$P_{\\mu_2}$', '$P_X$']
    colors = ['red', 'purple', 'orange', 'green', 'blue']

    for i, nx in enumerate(nx_values):
        ax = axes[i]
        data = all_data[nx]
        
        fiber_lengths_opt = data['fiber_lengths']
        # opt_params = data['optimized_params_array']
        # pred_params = data.get('predicted_params_array', None)
        # fiber_lengths_pred = data.get('fiber_lengths_pred', fiber_lengths_opt)
        
        fiber_lengths_opt = all_data[nx]['fiber_lengths']
        optimized_params_array = all_data[nx]['optimized_params_array']
        
        predicted_params_array = all_data[nx].get('predicted_params_array', None)
        fiber_lengths_pred = all_data[nx].get('fiber_lengths_pred', fiber_lengths_opt)  # ← NEW!

        # # Plot each parameter
        # for p_idx, (label, color) in enumerate(zip(param_labels, colors)):
        #     # Optimized (always available)
        #     if len(opt_params) > 0 and opt_params.shape[1] > p_idx:
        #         mask_opt = ~np.isnan(opt_params[:, p_idx])
        #         if np.any(mask_opt):
        #             ax.plot(fiber_lengths_opt[mask_opt], opt_params[mask_opt, p_idx], 
        #                    color=color, linestyle='-', alpha=0.4, linewidth=1.5,
        #                    label=f'Opt {label}' if i == 0 else "")
        # Plot parameters
        for param_idx, (label, color) in enumerate(zip(param_labels, colors)):
            # Optimized
            # ax.plot(fiber_lengths_opt, optimized_params_array[:, param_idx], color=color, linestyle='-', linewidth=2.0)
            ax.plot(fiber_lengths_opt, optimized_params_array[:, param_idx], 
            color=color, linestyle='-', linewidth=2.0, alpha=0.7,
            label=f'Opt {label}' if i == 0 else "")  #
            # # Predicted (may be empty)
            # if pred_params is not None and len(pred_params) > 0 and pred_params.shape[1] > p_idx:
            #     mask_pred = ~np.isnan(pred_params[:, p_idx])
            #     if np.any(mask_pred):
            #         ax.plot(fiber_lengths_pred[mask_pred], pred_params[mask_pred, p_idx], 
            #                color=color, linestyle='--', linewidth=2.5, alpha=0.8,
            #                label=f'Pred {label}' if i == 0 else "")
            # Predicted
            # if predicted_params_array is not None and len(predicted_params_array) > 0:
            #     ax.plot(fiber_lengths_pred, predicted_params_array[:, param_idx], color=color, linestyle=':', linewidth=4.0, alpha=0.7)  # ← Use fiber_lengths_pred!
            if predicted_params_array is not None and len(predicted_params_array) > 0:
                ax.plot(fiber_lengths_pred, predicted_params_array[:, param_idx], 
                color=color, linestyle=':', linewidth=4.0, alpha=0.7,
                label=f'Pred {label}' if i == 0 else "") 
        exponent = int(np.log10(float(nx)))
        ax.set_title(f"$n_X = 10^{{{exponent}}}$", fontsize=12)
        ax.set_xlabel("Fiber Length (km)")
        ax.set_ylabel("Parameter Value")
        ax.set_ylim(-0.05, 1.05)
        ax.grid(True, linestyle=':', alpha=0.4)
        
        if i == 0:
            ax.legend(loc='upper right', ncol=2, fontsize=8)

    for i in range(num_plots, len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Parameter subplots saved to {filename}")


In [13]:
def plot_relative_errors(fiber_lengths, optimized_key_rates, optimized_params_array, 
                         predicted_key_rates, predicted_params_array, epoch, filename, 
                         nx=None, threshold=1e-7):  # Increase threshold to 1e-7
    # Start timing
    plot_start_time = time.time()

    valid_mask = ~(np.isnan(optimized_key_rates) | np.isnan(predicted_key_rates))
    fiber_lengths = fiber_lengths[valid_mask]
    optimized_key_rates = optimized_key_rates[valid_mask]
    predicted_key_rates = predicted_key_rates[valid_mask]
    optimized_params_array = optimized_params_array[valid_mask]
    predicted_params_array = predicted_params_array[valid_mask]

    # Find cutoff where optimized key rate becomes very small
    cutoff_idx = np.where(optimized_key_rates <= threshold)[0]
    if len(cutoff_idx) > 0:
        cutoff_idx = cutoff_idx[0]
    else:
        cutoff_idx = len(fiber_lengths)

    fiber_lengths = fiber_lengths[:cutoff_idx]
    optimized_key_rates = optimized_key_rates[:cutoff_idx]
    predicted_key_rates = predicted_key_rates[:cutoff_idx]
    optimized_params_array = optimized_params_array[:cutoff_idx]
    predicted_params_array = predicted_params_array[:cutoff_idx]

    # Compute relative errors
    # Parameters: mu_1, mu_2, P_mu_1, P_mu_2, P_X
    relative_errors = []
    param_labels = ['$\mu_1$', '$\mu_2$', '$P_{\mu_1}$', '$P_{\mu_2}$', '$P_X$']
    for i in range(5):
        optimized = optimized_params_array[:, i]
        predicted = predicted_params_array[:, i]
        denominator = np.maximum(optimized, 1e-10)
        rel_error = (predicted - optimized) / denominator
        relative_errors.append(rel_error)
    
    # Key rate relative error with a higher threshold
    denominator = np.maximum(optimized_key_rates, 1e-7)  # Increase threshold to 1e-7
    key_rate_rel_error = (predicted_key_rates - optimized_key_rates) / denominator

    # Create figure with 6 subplots (2 rows, 3 columns)
    fig = plt.figure(figsize=(18, 10))
    gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

    # Plot relative errors
    for i in range(5):
        row = i // 3
        col = i % 3
        ax = fig.add_subplot(gs[row, col])
        ax.plot(fiber_lengths, relative_errors[i], 'b-', label=f'Relative Error {param_labels[i]}')
        ax.set_xlabel('Fiber Length (km)')
        ax.set_ylabel('Relative Error')
        ax.set_title(f'Relative Error for {param_labels[i]}')
        ax.grid(True)
        ax.legend(loc='best')

    # Key rate subplot
    ax_key = fig.add_subplot(gs[1, 2])
    ax_key.plot(fiber_lengths, key_rate_rel_error, 'r-', label='Relative Error Key Rate')
    ax_key.set_xlabel('Fiber Length (km)')
    ax_key.set_ylabel('Relative Error')
    ax_key.set_title('Relative Error for Key Rate')
    ax_key.grid(True)
    ax_key.legend(loc='best')

    # Overall title
    exponent = int(np.log10(nx)) if nx is not None else ''
    fig.suptitle(f'Relative Errors of Parameters for $n_X = 10^{{{exponent}}}$', fontsize=16)

    # Save and close
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Relative error plot saved to {filename}")

    # Print plotting time
    plot_time = time.time() - plot_start_time
    print(f"--- Plotting Time: {plot_time:.2f} seconds ---")

In [14]:
# Training loop
import time

# Training loop
total_start_time = time.time()
num_epochs = 5000
for epoch in range(num_epochs):
    start_time = time.time()

    # Training phase
    model.train()
    running_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_loss)

    # Validation phase
    model.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_running_loss += loss.item() * inputs.size(0)
    val_loss = val_running_loss / len(val_loader.dataset)
    val_losses.append(val_loss)

    # Update learning rate
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    learning_rates.append(current_lr)

    # Evaluate and plot at the first and last epochs
    eval_time = 0
    plot_time = 0
    if epoch == 0 or epoch == num_epochs - 1:
        model.eval()
        with torch.no_grad():
            # Prepare data for all n_X values
            plot_data = {}
            eval_start = time.time()
            for nx, eval_data in all_evaluation_data.items():
                fiber_lengths = eval_data['fiber_lengths']
                optimized_params_array = eval_data['optimized_params_array']
                optimized_key_rates = eval_data['optimized_key_rates']
                X_test_tensor = eval_data['X_test_tensor']

                # Compute predicted parameters and key rates
                # predicted_params_scaled = model(X_test_tensor).cpu().numpy()
                predicted_params_scaled = model(X_test_tensor).detach().cpu().numpy()
                predicted_params_scaled = np.clip(predicted_params_scaled, 0, 1)
                predicted_params_array = y_scaler.inverse_transform(predicted_params_scaled)
                predicted_params_array[:, 0] = np.maximum(predicted_params_array[:, 0], 1e-6) # mu_1
                predicted_params_array[:, 1] = np.maximum(predicted_params_array[:, 1], 1e-6) # mu_2
                predicted_params_array[:, 2] = np.clip(predicted_params_array[:, 2], 0, 1)    # P_mu_1
                predicted_params_array[:, 3] = np.clip(predicted_params_array[:, 3], 0, 1)    # P_mu_2
                predicted_params_array[:, 4] = np.clip(predicted_params_array[:, 4], 0, 1)    # P_X
                
                # --- NEW: Enforce Sum Constraint ---
                prob_sum = predicted_params_array[:, 2] + predicted_params_array[:, 3]
                # Option A: Normalize if sum > 1
                mask_sum_gt_1 = prob_sum > 1.0
                if np.any(mask_sum_gt_1):
                    predicted_params_array[mask_sum_gt_1, 2] /= prob_sum[mask_sum_gt_1] # Scale P_mu_1
                    predicted_params_array[mask_sum_gt_1, 3] /= prob_sum[mask_sum_gt_1] # Scale P_mu_2
                # Option B: Clip individual probabilities again (less physically motivated, but simpler)
                # predicted_params_array[:, 2] = np.minimum(predicted_params_array[:, 2], 1.0 - predicted_params_array[:, 3]) # Ensure P_mu_1 <= 1 - P_mu_2
                
                # Compute and filter predicted key rates
                predicted_key_rates = []
                valid_indices = []
                for idx, (params, L) in enumerate(zip(predicted_params_array, fiber_lengths)):
                    key_rate = safe_objective(params, L, nx, alpha=0.2, eta_Bob=0.1, P_dc_value=6e-7, 
                                              epsilon_sec=1e-10, epsilon_cor=1e-15, f_EC=1.16, 
                                              e_mis=5e-3, P_ap=0, n_event=1)
                    if key_rate is None:
                        print(f"Skipping invalid predicted key rate at index {idx} for n_X = {nx}")
                        continue
                    predicted_key_rates.append(key_rate)
                    valid_indices.append(idx)
                predicted_key_rates = np.array(predicted_key_rates)

                # Store data WITHOUT coupling optimized and predicted filtering
                # This ensures optimized data is always available even when predictions fail
                
                # Keep predicted data only for valid indices
                valid_predicted_params = predicted_params_array[valid_indices] if len(valid_indices) > 0 else np.array([])
                
                # Match fiber_lengths for predicted data
                valid_fiber_lengths_pred = fiber_lengths[valid_indices] if len(valid_indices) > 0 else np.array([])
                
                # Store data for plotting - DECOUPLED
                # plot_data[nx] = {
                #     'fiber_lengths': fiber_lengths,  # KEEP ALL for optimized data
                #     'optimized_key_rates': optimized_key_rates,  # KEEP ALL
                #     'optimized_params_array': optimized_params_array,  # KEEP ALL
                #     'predicted_key_rates': predicted_key_rates,  # Only valid predictions
                #     'predicted_params_array': valid_predicted_params,  # Only valid predictions
                #     'fiber_lengths_pred': valid_fiber_lengths_pred  # Match predicted data points
                # }

                plot_data[nx] = {
                    'fiber_lengths': fiber_lengths,  # ← KEEP ALL (for optimized)
                    'optimized_key_rates': optimized_key_rates,  # ← KEEP ALL
                    'optimized_params_array': optimized_params_array,  # ← KEEP ALL
                    'predicted_key_rates': predicted_key_rates,  # ← Only valid predictions
                    'predicted_params_array': predicted_params_array[valid_indices] if len(valid_indices) > 0 else np.array([]),  # ← Only valid predictions
                    'fiber_lengths_pred': fiber_lengths[valid_indices] if len(valid_indices) > 0 else np.array([])  # ← Matching fiber lengths for predictions
                }
                # Store data for plotting

                # Plot relative errors for this n_X
                # plot_relative_errors(
                #     fiber_lengths, optimized_key_rates, optimized_params_array,
                #     predicted_key_rates, predicted_params_array, epoch,
                #     f'parameter_relative_error_nx_{nx:.0e}.png', nx=nx
                # )
                plot_relative_errors(
                    valid_fiber_lengths_pred, 
                    optimized_key_rates[valid_indices], 
                    optimized_params_array[valid_indices],  # ← Use filtered versions
                    predicted_key_rates, 
                    predicted_params_array[valid_indices],  # Use this instead # ← Use filtered versions
                    epoch,
                    f'parameter_relative_error_nx_{nx:.0e}.png', 
                    nx=nx
                )

            eval_time += time.time() - eval_start

            # Plot subplots at the first and last epochs
            plot_start = time.time()
            if epoch == 0:
                plot_keyrate_subplots(plot_data, epoch, f'keyrate_subplots_first_epoch.png')
                plot_parameters_subplots(plot_data, epoch, f'parameters_subplots_first_epoch.png')
            if epoch == num_epochs - 1:
                plot_keyrate_subplots(plot_data, epoch, f'keyrate_subplots_last_epoch.png')
                plot_parameters_subplots(plot_data, epoch, f'parameters_subplots_last_epoch.png')
            plot_time += time.time() - plot_start

            torch.save(model.state_dict(), 'models/bb84_nn_model.pth')
            print("Model saved to bb84_nn_model.pth")

    # Print epoch results with timing and learning rate
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss:a {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, "
          f"Learning Rate: {current_lr:.6f}, Time: {epoch_time:.2f}s (Eval: {eval_time:.2f}s, Plot: {plot_time:.2f}s)")

# Print total training time
total_training_time = time.time() - total_start_time
print(f"--- Total Training Time: {total_training_time:.2f} seconds ---")

Objective returned non-positive key rate (-1e+250) for params [0.5185131  0.20677878 0.07785834 0.53687537 0.53049225], L=0.0, nx=10000
Skipping invalid predicted key rate at index 0 for n_X = 10000
Objective returned non-positive key rate (-1e+250) for params [0.51851237 0.20662959 0.07774336 0.5365604  0.5302074 ], L=0.2002002002002002, nx=10000
Skipping invalid predicted key rate at index 1 for n_X = 10000
Objective returned non-positive key rate (-1e+250) for params [0.5185117  0.20648037 0.07762836 0.53624547 0.5299225 ], L=0.4004004004004004, nx=10000
Skipping invalid predicted key rate at index 2 for n_X = 10000
Objective returned non-positive key rate (-1e+250) for params [0.518511   0.20633115 0.07751341 0.53593045 0.52963763], L=0.6006006006006006, nx=10000
Skipping invalid predicted key rate at index 3 for n_X = 10000
Objective returned non-positive key rate (-1e+250) for params [0.51851034 0.20618196 0.07739843 0.53561544 0.5293527 ], L=0.8008008008008008, nx=10000
Skipping

/var/folders/8m/0wg1hssn6n79tjc8mh6p_spr0000gn/T/ipykernel_45184/4027905518.py:72: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.95])


Relative error plot saved to parameter_relative_error_nx_1e+04.png
--- Plotting Time: 0.68 seconds ---
Objective returned non-positive key rate (-1e+250) for params [0.5147361  0.23466882 0.10582416 0.5877972  0.5824983 ], L=0.0, nx=100000
Skipping invalid predicted key rate at index 0 for n_X = 100000
Objective returned non-positive key rate (-1e+250) for params [0.5147124  0.23448004 0.10566363 0.587324   0.5820697 ], L=0.2002002002002002, nx=100000
Skipping invalid predicted key rate at index 1 for n_X = 100000
Objective returned non-positive key rate (-1e+250) for params [0.5146887  0.23429126 0.10550313 0.5868507  0.581641  ], L=0.4004004004004004, nx=100000
Skipping invalid predicted key rate at index 2 for n_X = 100000
Objective returned non-positive key rate (-1e+250) for params [0.5146649  0.23410246 0.10534259 0.58637744 0.5812124 ], L=0.6006006006006006, nx=100000
Skipping invalid predicted key rate at index 3 for n_X = 100000
Objective returned non-positive key rate (-1e+25

/var/folders/8m/0wg1hssn6n79tjc8mh6p_spr0000gn/T/ipykernel_45184/4027905518.py:72: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.95])


Relative error plot saved to parameter_relative_error_nx_1e+05.png
--- Plotting Time: 0.59 seconds ---
Objective returned non-positive key rate (-1e+250) for params [0.51349604 0.2748099  0.14580823 0.6738576  0.67424685], L=0.0, nx=1000000
Skipping invalid predicted key rate at index 0 for n_X = 1000000
Objective returned non-positive key rate (-1e+250) for params [0.5134904  0.27462786 0.1457964  0.673529   0.673931  ], L=0.2002002002002002, nx=1000000
Skipping invalid predicted key rate at index 1 for n_X = 1000000
Objective returned non-positive key rate (-1e+250) for params [0.5134847  0.2744458  0.14578451 0.6732003  0.67361504], L=0.4004004004004004, nx=1000000
Skipping invalid predicted key rate at index 2 for n_X = 1000000
Objective returned non-positive key rate (-1e+250) for params [0.51347905 0.27426377 0.14577262 0.6728717  0.67329913], L=0.6006006006006006, nx=1000000
Skipping invalid predicted key rate at index 3 for n_X = 1000000
Objective returned non-positive key rate

...

Relative error plot saved to parameter_relative_error_nx_1e+07.png
--- Plotting Time: 0.60 seconds ---
Objective returned non-positive key rate (-1e+250) for params [0.50438815 0.34280032 0.16910379 0.77247816 0.82911235], L=0.0, nx=100000000
Skipping invalid predicted key rate at index 0 for n_X = 100000000
Objective returned non-positive key rate (-1e+250) for params [0.50427395 0.34272793 0.16908519 0.7724117  0.82900876], L=0.2002002002002002, nx=100000000
Skipping invalid predicted key rate at index 1 for n_X = 100000000
Objective returned non-positive key rate (-1e+250) for params [0.5040455  0.34258306 0.16904797 0.7722787  0.8288017 ], L=0.6006006006006006, nx=100000000
Skipping invalid predicted key rate at index 3 for n_X = 100000000
Objective returned non-positive key rate (-1e+250) for params [0.5039312  0.3425106  0.16902937 0.7722122  0.8286983 ], L=0.8008008008008008, nx=100000000
Skipping invalid predicted key rate at index 4 for n_X = 100000000
Objective returned non-p

/var/folders/8m/0wg1hssn6n79tjc8mh6p_spr0000gn/T/ipykernel_45184/4027905518.py:72: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.95])


Relative error plot saved to parameter_relative_error_nx_1e+08.png
--- Plotting Time: 0.58 seconds ---
Objective returned non-positive key rate (-1e+250) for params [0.48815134 0.36582395 0.18793939 0.77275324 0.8765765 ], L=0.0, nx=1000000000
Skipping invalid predicted key rate at index 0 for n_X = 1000000000
Objective returned non-positive key rate (-1e+250) for params [0.4878682  0.3658054  0.18735732 0.77275324 0.8765765 ], L=0.4004004004004004, nx=1000000000
Skipping invalid predicted key rate at index 2 for n_X = 1000000000
Objective returned non-positive key rate (-1e+250) for params [0.48776105 0.36572072 0.18732122 0.77275324 0.8765765 ], L=0.6006006006006006, nx=1000000000
Skipping invalid predicted key rate at index 3 for n_X = 1000000000
Objective returned non-positive key rate (-1e+250) for params [0.48765388 0.36563602 0.18728496 0.77275324 0.8765765 ], L=0.8008008008008008, nx=1000000000
Skipping invalid predicted key rate at index 4 for n_X = 1000000000
Objective return

/var/folders/8m/0wg1hssn6n79tjc8mh6p_spr0000gn/T/ipykernel_45184/4027905518.py:72: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.95])


Relative error plot saved to parameter_relative_error_nx_1e+09.png
--- Plotting Time: 0.58 seconds ---
✓ Key rate subplots saved to keyrate_subplots_last_epoch.png
✓ Parameter subplots saved to parameters_subplots_last_epoch.png
Model saved to bb84_nn_model.pth
Epoch 5000/5000, Train Loss:a 0.0003, Val Loss: 0.0002, Learning Rate: 0.000001, Time: 6.50s (Eval: 4.74s, Plot: 1.53s)
--- Total Training Time: 1170.75 seconds ---


In [15]:
# %%
# Plotting the losses with smoothing
def moving_average(data, window_size=5):
    return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

epochs = range(1, len(train_losses) + 1)
window_size = 5
smoothed_train_losses = moving_average(train_losses, window_size)
smoothed_val_losses = moving_average(val_losses, window_size)
smoothed_epochs = range(window_size, len(train_losses) + 1)

In [16]:
plt.figure(figsize=(12, 6))
# plt.plot(smoothed_epochs, smoothed_train_losses, label='Smoothed Training Loss', linestyle='-')
# plt.plot(smoothed_epochs, smoothed_val_losses, label='Smoothed Validation Loss', linestyle='--')
plt.plot(epochs, train_losses, label='Training Loss', linestyle='-', alpha=0.3)
plt.plot(epochs, val_losses, label='Validation Loss', linestyle='--', alpha=0.5)
plt.xlabel('Epochs')
plt.ylim(0, 0.002)
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.grid(True)
plt.tight_layout()
plt.savefig('loss_plot.png', dpi=150)
plt.show()
# plt.close()

print("Training Complete")

# Plotting the learning rates
plt.figure(figsize=(12, 6))
plt.plot(epochs, learning_rates, label='Learning Rate', linestyle='-', color='tab:blue')
plt.xlabel('Epochs')
plt.ylabel('Learning Rate')
plt.legend()
plt.title('Learning Rate over Epochs')
plt.grid(True)
plt.tight_layout()
plt.savefig('learning_rate_plot.png', dpi=150)
plt.show()
# plt.close()

Training Complete


/var/folders/8m/0wg1hssn6n79tjc8mh6p_spr0000gn/T/ipykernel_45184/2391135169.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/8m/0wg1hssn6n79tjc8mh6p_spr0000gn/T/ipykernel_45184/2391135169.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
def plot_keyrate_and_parameters(fiber_lengths, optimized_key_rates, optimized_params_array, 
                                predicted_key_rates, predicted_params_array, epoch, filename, 
                                learning_rates=None, nx=None):
    """
    Plots key rates and parameters for a specific n_X value.
    
    Parameters:
    - fiber_lengths: Array of fiber lengths.
    - optimized_key_rates: Array of optimized key rates.
    - optimized_params_array: Array of optimized parameters [mu_1, mu_2, P_mu_1, P_mu_2, P_X].
    - predicted_key_rates: Array of predicted key rates.
    - predicted_params_array: Array of predicted parameters [mu_1, mu_2, P_mu_1, P_mu_2, P_X].
    - epoch: Epoch number or string (e.g., 'final_test').
    - filename: Output filename for the plot.
    - learning_rates: List of learning rates over epochs (optional, not used).
    - nx: The n_X value for the plot title (optional).
    """

    # Start timing
    plot_start_time = time.time()

    # Create a figure with 2 subplots: key rate (left), all parameters (right)
    fig = plt.figure(figsize=(15, 6))  # Adjusted height for horizontal layout

    # Define the grid layout: 1 row, 2 columns
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 1])

    # Key Rate Plot (Left)
    ax_keyrate = fig.add_subplot(gs[0, 0])

    # Filter out NaN values
    valid_mask = ~(np.isnan(optimized_key_rates) | np.isnan(predicted_key_rates))
    fiber_lengths = fiber_lengths[valid_mask]
    optimized_key_rates = optimized_key_rates[valid_mask]
    predicted_key_rates = predicted_key_rates[valid_mask]
    optimized_params_array = optimized_params_array[valid_mask]
    predicted_params_array = predicted_params_array[valid_mask]

    # Find cutoff where optimized key rate becomes very small
    threshold = 1e-8
    cutoff_idx = np.where(optimized_key_rates <= threshold)[0]
    if len(cutoff_idx) > 0:
        cutoff_idx = cutoff_idx[0]
    else:
        cutoff_idx = len(fiber_lengths)

    fiber_lengths = fiber_lengths[:cutoff_idx]
    optimized_key_rates = optimized_key_rates[:cutoff_idx]
    predicted_key_rates = predicted_key_rates[:cutoff_idx]
    optimized_params_array = optimized_params_array[:cutoff_idx]
    predicted_params_array = predicted_params_array[:cutoff_idx]

    # Plot key rates with distinct linestyles and increased thickness for predicted
    ax_keyrate.plot(fiber_lengths, np.log10(predicted_key_rates), 'r--', label='Predicted Key Rate', linewidth=4.0, alpha=0.7)
    ax_keyrate.plot(fiber_lengths, np.log10(optimized_key_rates), 'b-', label='Optimized Key Rate', linewidth=2.0, alpha=1.0)

    # Use scientific notation for nx in the title
    ax_keyrate.set_title(f"Key Rates for $n_X = 5 \\times 10^8$")  # Fixed title
    ax_keyrate.set_xlabel("Fiber Length (km)")
    ax_keyrate.set_ylabel("Secret Key Rate per Pulse")
    ax_keyrate.grid(True)
    ax_keyrate.legend(loc='upper right')

    # Parameter Plot (Right) - Combine probabilities and intensities
    ax_params = fig.add_subplot(gs[0, 1])

    param_labels = ['$\mu_1$', '$\mu_2$', '$P_{\mu_1}$', '$P_{\mu_2}$', '$P_X$']
    colors = ['red', 'purple', 'orange', 'green', 'blue']
    param_indices = [0, 1, 2, 3, 4]  # Indices for mu_1, mu_2, P_mu_1, P_mu_2, P_X  

    # Plot all parameters
    for param_idx, (label, color) in zip(param_indices, zip(param_labels, colors)):
        ax_params.plot(fiber_lengths, predicted_params_array[:, param_idx], label=f'Predicted {label}', color=color, linestyle='--', linewidth=4.0, alpha=0.7)
        ax_params.plot(fiber_lengths, optimized_params_array[:, param_idx], label=f'Optimized {label}', color=color, linestyle='-', linewidth=2.0, alpha=1.0)
        
    # Use scientific notation for nx in the title
    ax_params.set_title(f"Parameters for $n_X = 5 \\times 10^8$")  # Fixed title
    ax_params.set_xlabel("Fiber Length (km)")
    ax_params.set_ylabel("Parameter Value")
    ax_params.set_ylim(0.0, 1.0)
    ax_params.grid(True)

    # Move legend outside the plot to avoid overlap
    ax_params.legend(loc='center left', bbox_to_anchor=(1, 0.5))

    # Adjust layout to prevent overlap
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Comparison plot saved to {filename}")

In [19]:
# for nx in all_evaluation_data.keys():
#     eval_data = all_evaluation_data[nx]
#     fiber_lengths = eval_data['fiber_lengths']
#     optimized_params_array = eval_data['optimized_params_array']
#     optimized_key_rates = eval_data['optimized_key_rates']
#     X_test_tensor = eval_data['X_test_tensor']
#     with torch.no_grad():
#         predicted_params_scaled = model(X_test_tensor).cpu().numpy()
#         predicted_params_scaled = np.clip(predicted_params_scaled, 0, 1)
#         predicted_params_array = y_scaler.inverse_transform(predicted_params_scaled)
        
#         # Apply constraints
#         predicted_params_array[:, 0] = np.maximum(predicted_params_array[:, 0], 1e-6)  # mu_1
#         predicted_params_array[:, 1] = np.maximum(predicted_params_array[:, 1], 1e-6)  # mu_2
#         predicted_params_array[:, 2] = np.clip(predicted_params_array[:, 2], 0, 1)  # P_mu_1
#         predicted_params_array[:, 3] = np.clip(predicted_params_array[:, 3], 0, 1)  # P_mu_2
#         predicted_params_array[:, 4] = np.clip(predicted_params_array[:, 4], 0, 1)  # P_X
        
#         # Enforce sum constraint
#         prob_sum = predicted_params_array[:, 2] + predicted_params_array[:, 3]
#         mask_sum_gt_1 = prob_sum > 1.0
#         if np.any(mask_sum_gt_1):

# # for nx in all_evaluation_data.keys():
# #     eval_data = all_evaluation_data[nx]
# #     fiber_lengths = eval_data['fiber_lengths']
# #     optimized_params_array = eval_data['optimized_params_array']
# #     optimized_key_rates = eval_data['optimized_key_rates']
# #     X_test_tensor = eval_data['X_test_tensor']

# #     with torch.no_grad():
# #         predicted_params_scaled = model(X_test_tensor).cpu().numpy()
# #         predicted_params_scaled = np.clip(predicted_params_scaled, 0, 1)
# #         predicted_params_array = y_scaler.inverse_transform(predicted_params_scaled)
# #         predicted_params_array[:, 0] = np.maximum(predicted_params_array[:, 0], 1e-6)
# #         predicted_params_array[:, 1] = np.maximum(predicted_params_array[:, 1], 1e-6)
# #         predicted_params_array[:, 2] = np.clip(predicted_params_array[:, 2], 0, 1)
# #         predicted_params_array[:, 3] = np.clip(predicted_params_array[:, 3], 0, 1)
# #         predicted_params_array[:, 4] = np.clip(predicted_params_array[:, 4], 0, 1)

#         # Compute and filter predicted key rates
#         # Compute and filter predicted key rates
#         predicted_key_rates = []
#         valid_indices = []
#         for idx, (params, L) in enumerate(zip(predicted_params_array, fiber_lengths)):
#             key_rate = safe_objective(params, L, nx, alpha=0.2, eta_Bob=0.1, P_dc_value=6e-7, 
#                                       epsilon_sec=1e-10, epsilon_cor=1e-15, f_EC=1.16, 
#                                       e_mis=5e-3, P_ap=0, n_event=1)
#             if key_rate is None:
#                 print(f"Skipping invalid predicted key rate at index {idx} for n_X = {nx}")
#                 continue
#             predicted_key_rates.append(key_rate)
#             valid_indices.append(idx)
#         predicted_key_rates = np.array(predicted_key_rates)
#         # After computing valid_indices for predictions
#         if len(valid_indices) > 0:
#             plot_relative_errors(
#                 fiber_lengths[valid_indices],  # Only plot where we have predictions
#                 optimized_key_rates[valid_indices],  # Matching optimized data
#                 optimized_params_array[valid_indices],  # Matching optimized params
#                 predicted_key_rates,  # Already filtered
#                 valid_predicted_params, epoch,
#                 f'parameter_relative_error_nx_{nx:.0e}.png', nx=nx
#             )
#     else:
#         print(f"⚠️  No valid predictions for n_X = {nx}, skipping relative error plot")
#     #     predicted_key_rates = []
#     #     valid_indices = []
#     #     for idx, (params, L) in enumerate(zip(predicted_params_array, fiber_lengths)):
#     #         key_rate = safe_objective(params, L, nx, alpha=0.2, eta_Bob=0.1, P_dc_value=6e-7, 
#     #                                   epsilon_sec=1e-10, epsilon_cor=1e-15, f_EC=1.16, 
#     #                                   e_mis=5e-3, P_ap=0, n_event=1)
#     #         if key_rate is None:
#     #             print(f"Skipping invalid predicted key rate at index {idx} for n_X = {nx}")
#     #             continue
#     #         predicted_key_rates.append(key_rate)
#     #         valid_indices.append(idx)
#     #     predicted_key_rates = np.array(predicted_key_rates)

#     #     # Filter data based on valid predicted key rates
#     #     fiber_lengths = fiber_lengths[valid_indices]
#     #     # optimized_key_rates = optimized_key_rates[valid_indices]
#     #     optimized_params_array = optimized_params_array[valid_indices]
#     #     predicted_params_array = predicted_params_array[valid_indices]

#     # # Plot relative errors for this n_X
#     # plot_start_time = time.time()
#     # plot_relative_errors(
#     #     fiber_lengths, optimized_key_rates, optimized_params_array,
#     #     predicted_key_rates, predicted_params_array, epoch='final_test',
#     #     filename=f'relative_error_nx_{nx:.0e}.png', nx=nx
#     # )
#     # print(f"--- Final Test Plotting Time (relative_error_nx_{nx:.0e}): {time.time() - plot_start_time:.2f} seconds ---")


for nx in all_evaluation_data.keys():
    eval_data = all_evaluation_data[nx]
    fiber_lengths = eval_data['fiber_lengths']
    optimized_params_array = eval_data['optimized_params_array']
    optimized_key_rates = eval_data['optimized_key_rates']
    X_test_tensor = eval_data['X_test_tensor']
    
    with torch.no_grad():
        predicted_params_scaled = model(X_test_tensor).cpu().numpy()
        predicted_params_scaled = np.clip(predicted_params_scaled, 0, 1)
        predicted_params_array = y_scaler.inverse_transform(predicted_params_scaled)
        
        # Apply constraints
        predicted_params_array[:, 0] = np.maximum(predicted_params_array[:, 0], 1e-6)  # mu_1
        predicted_params_array[:, 1] = np.maximum(predicted_params_array[:, 1], 1e-6)  # mu_2
        predicted_params_array[:, 2] = np.clip(predicted_params_array[:, 2], 0, 1)  # P_mu_1
        predicted_params_array[:, 3] = np.clip(predicted_params_array[:, 3], 0, 1)  # P_mu_2
        predicted_params_array[:, 4] = np.clip(predicted_params_array[:, 4], 0, 1)  # P_X
        
        # Enforce sum constraint
        prob_sum = predicted_params_array[:, 2] + predicted_params_array[:, 3]
        mask_sum_gt_1 = prob_sum > 1.0
        if np.any(mask_sum_gt_1):
            predicted_params_array[mask_sum_gt_1, 2] /= prob_sum[mask_sum_gt_1]
            predicted_params_array[mask_sum_gt_1, 3] /= prob_sum[mask_sum_gt_1]

        # Compute and filter predicted key rates
        predicted_key_rates = []
        valid_indices = []
        for idx, (params, L) in enumerate(zip(predicted_params_array, fiber_lengths)):
            key_rate = safe_objective(params, L, nx, alpha=0.2, eta_Bob=0.1, P_dc_value=6e-7, 
                                      epsilon_sec=1e-10, epsilon_cor=1e-15, f_EC=1.16, 
                                      e_mis=5e-3, P_ap=0, n_event=1)
            if key_rate is None:
                print(f"Skipping invalid predicted key rate at index {idx} for n_X = {nx}")
                continue
            predicted_key_rates.append(key_rate)
            valid_indices.append(idx)
        predicted_key_rates = np.array(predicted_key_rates)
        
        # After computing valid_indices for predictions
        if len(valid_indices) > 0:
                plot_relative_errors(fiber_lengths[valid_indices], optimized_key_rates[valid_indices], optimized_params_array[valid_indices], predicted_key_rates, predicted_params_array[valid_indices], epoch='final_test', f'parameter_relative_error_nx_{nx:.0e}.png', nx=nx)
            else:
                print(f"⚠️  No valid predictions for n_X = {nx}, skipping relative error plot")

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 139)

In [ ]:
def plot_relative_errors(fiber_lengths, optimized_key_rates, optimized_params_array, 
                        predicted_key_rates, predicted_params_array, epoch, filename, 
                        nx=None, threshold=1e-8):
    """
    Plots relative errors of predicted vs. optimized parameters and key rate for a specific n_X.
    
    Parameters:
    - fiber_lengths: Array of fiber lengths (km).
    - optimized_key_rates: Array of optimized key rates.
    - optimized_params_array: Array of optimized parameters [mu_1, mu_2, P_mu_1, P_mu_2, P_X].
    - predicted_key_rates: Array of predicted key rates.
    - predicted_params_array: Array of predicted parameters [mu_1, mu_2, P_mu_1, P_mu_2, P_X].
    - epoch: Epoch number or string (e.g., 'final_test').
    - filename: Output filename for the plot.
    - nx: The n_X value for the plot title (optional).
    - threshold: Key rate threshold for physical cutoff (default: 1e-8).
    """
    # Start timing
    plot_start_time = time.time()

    # Filter out NaN values
    valid_mask = ~(np.isnan(optimized_key_rates) | np.isnan(predicted_key_rates))
    fiber_lengths = fiber_lengths[valid_mask]
    optimized_key_rates = optimized_key_rates[valid_mask]
    predicted_key_rates = predicted_key_rates[valid_mask]
    optimized_params_array = optimized_params_array[valid_mask]
    predicted_params_array = predicted_params_array[valid_mask]

    # Find cutoff where optimized key rate becomes very small
    cutoff_idx = np.where(optimized_key_rates <= threshold)[0]
    if len(cutoff_idx) > 0:
        cutoff_idx = cutoff_idx[0]
    else:
        cutoff_idx = len(fiber_lengths)

    fiber_lengths = fiber_lengths[:cutoff_idx]
    optimized_key_rates = optimized_key_rates[:cutoff_idx]
    predicted_key_rates = predicted_key_rates[:cutoff_idx]
    optimized_params_array = optimized_params_array[:cutoff_idx]
    predicted_params_array = predicted_params_array[:cutoff_idx]

    # Compute relative errors
    # Parameters: mu_1, mu_2, P_mu_1, P_mu_2, P_X
    relative_errors = []
    param_labels = ['$\mu_1$', '$\mu_2$', '$P_{\mu_1}$', '$P_{\mu_2}$', '$P_X$']
    for i in range(5):
        optimized = optimized_params_array[:, i]
        predicted = predicted_params_array[:, i]
        # Avoid division by zero with a small threshold
        denominator = np.maximum(optimized, 1e-10)
        rel_error = (predicted - optimized) / denominator
        relative_errors.append(rel_error)
    
    # Key rate relative error
    denominator = np.maximum(optimized_key_rates, 1e-10)
    key_rate_rel_error = (predicted_key_rates - optimized_key_rates) / denominator

    # Create figure with 6 subplots (2 rows, 3 columns)
    fig = plt.figure(figsize=(18, 10))
    gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

    # Plot relative errors
    for i in range(5):
        row = i // 3
        col = i % 3
        ax = fig.add_subplot(gs[row, col])
        ax.plot(fiber_lengths, relative_errors[i], 'b-', label=f'Relative Error {param_labels[i]}')
        ax.set_xlabel('Fiber Length (km)')
        ax.set_ylabel('Relative Error')
        ax.set_title(f'Relative Error for {param_labels[i]}')
        ax.grid(True)
        ax.legend(loc='best')

    # Key rate subplot
    ax_key = fig.add_subplot(gs[1, 2])
    ax_key.plot(fiber_lengths, key_rate_rel_error, 'r-', label='Relative Error Key Rate')
    ax_key.set_xlabel('Fiber Length (km)')
    ax_key.set_ylabel('Relative Error')
    ax_key.set_title('Relative Error for Key Rate')
    ax_key.grid(True)
    ax_key.legend(loc='best')

    # Overall title
    exponent = int(np.log10(nx)) if nx is not None else ''
        # Example title update (adjust based on actual function structure)
    fig.suptitle(f"Relative Errors of Parameters for $n_X = 5 \\times 10^8$", fontsize=16)

    # Save and close
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Relative error plot saved to {filename}")

    # Print plotting time
    plot_time = time.time() - plot_start_time
    print(f"--- Plotting Time: {plot_time:.2f} seconds ---")